# 📊 Preprocesamiento - Dataset Estudiantes

## 🎯 1. Objetivo del notebook

En este notebook se realiza el preprocesamiento inicial del dataset de estudiantes a partir de las conclusiones obtenidas en el análisis exploratorio.

El objetivo principal es preparar una versión limpia del conjunto de datos, tratando valores nulos, revisando duplicados y comprobando que las variables mantienen una estructura adecuada para las siguientes fases del proyecto.

Este paso es necesario antes de entrenar los modelos de regresión y clasificación, ya que los algoritmos de Machine Learning requieren datos consistentes y sin valores ausentes.

## 2. Importación de librerías y carga de datos

In [1]:
# A continuación, importamos las librerías necesarias para poder realizar el trabajo.

import pandas as pd
import numpy as np

from pathlib import Path

# Configuración para visualizar todas las columnas del DataFrame
pd.set_option("display.max_columns", None)

In [2]:
# Carga de las rutas de los archivos
ruta_raw = Path("../Data/raw/dataset_estudiantes.csv")
ruta_processed = Path("../Data/processed/dataset_estudiantes_limpio.csv")

# Carga del dataset original
df = pd.read_csv(ruta_raw)

# Mostramos las primeras filas
df.head()

,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,nivel_dificultad,tiene_tutor,horario_estudio_preferido,estilo_aprendizaje,nota_final,aprobado
0,8.957476,48.830601,86.640182,6.675694,25,Fácil,Sí,Tarde,Lectura/Escritura,84.4,1
1,11.042524,80.825707,83.449655,4.616844,18,Difícil,No,Tarde,NaN,72.0,1
2,4.510776,90.383694,74.623607,7.755246,25,Fácil,No,Mañana,Lectura/Escritura,80.0,1
3,6.647213,81.878257,82.849841,8.592826,23,Fácil,No,NaN,Visual,78.2,1
4,1.000000,66.254179,54.539935,6.671840,21,Medio,No,NaN,Auditivo,66.0,1


### Observaciones

Se importan las librerías necesarias para cargar, revisar y transformar el dataset.

En esta fase se utilizan principalmente `pandas` y `numpy`, ya que el objetivo es realizar tareas de limpieza y preparación de los datos.

También se utiliza `Path` para trabajar con rutas de archivos de forma más ordenada y reproducible.

Se carga el dataset original desde la carpeta `Data/raw`.

Es importante partir siempre de los datos originales para que el proceso de limpieza sea reproducible.  
A partir de este DataFrame se creará posteriormente una versión procesada que se guardará en la carpeta `Data/processed`.


## 3. Revisión inicial antes del preprocesamiento y creación de la copia del dataset


In [3]:
# Dimensiones del dataset original
print(f"El dataset original contiene {df.shape[0]} filas y {df.shape[1]} columnas.")

El dataset original contiene 1000 filas y 11 columnas.


In [4]:
# Información general del dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   horas_estudio_semanal      1000 non-null   float64
 1   nota_anterior              1000 non-null   float64
 2   tasa_asistencia            1000 non-null   float64
 3   horas_sueno                850 non-null    float64
 4   edad                       1000 non-null   int64  
 5   nivel_dificultad           1000 non-null   object 
 6   tiene_tutor                1000 non-null   object 
 7   horario_estudio_preferido  900 non-null    object 
 8   estilo_aprendizaje         950 non-null    object 
 9   nota_final                 1000 non-null   float64
 10  aprobado                   1000 non-null   int64  
dtypes: float64(5), int64(2), object(4)
memory usage: 86.1+ KB


In [5]:
# Revisión de valores nulos antes del preprocesamiento
valores_nulos_pre = pd.DataFrame({
    "valores_nulos": df.isnull().sum(),
    "porcentaje_nulos": (df.isnull().mean() * 100).round(2)
})

valores_nulos_pre[valores_nulos_pre["valores_nulos"] > 0]

,valores_nulos,porcentaje_nulos
horas_sueno,150,15.0
horario_estudio_preferido,100,10.0
estilo_aprendizaje,50,5.0


In [6]:
# Creamos una copia del dataset original para aplicar el preprocesamiento
df_clean = df.copy()

# Comprobamos las dimensiones de la copia
df_clean.shape

(1000, 11)

### Revisión inicial antes del preprocesamiento

Se carga el dataset original desde la carpeta `Data/raw`.

Es importante partir siempre de los datos originales para que el proceso de limpieza sea reproducible. A partir de este DataFrame se creará posteriormente una versión procesada que se guardará en la carpeta `Data/processed`.

Antes de aplicar transformaciones, se revisa de nuevo la estructura general del dataset.

Esta comprobación permite confirmar el número de registros y columnas, los tipos de datos y la presencia de valores nulos.

Según el análisis exploratorio anterior, se habían detectado valores ausentes en algunas variables, por lo que será necesario tratarlos antes de continuar con el modelado.

Una vez que hemos comprobado que los datos están correctos, creamos una copia para mantener el dataser original intacto. Todas las transformaciones de limpieza se realizarán sobre la copia, así si fuese necesarios revisar o repetir algún paso, se puede volver fácilmente al dataset original.

## 4. Tratamiento de valores nulos

En esta fase se tratan los valores ausentes detectados durante la revisión inicial del dataset.

Según el análisis exploratorio previo, las columnas con valores nulos son `horas_sueno`, `horario_estudio_preferido` y `estilo_aprendizaje`.

Para la variable numérica `horas_sueno`, se utilizará la mediana como valor de imputación, ya que es una medida robusta frente a posibles valores extremos.

En el caso de las variables categóricas `horario_estudio_preferido` y `estilo_aprendizaje`, los valores nulos se sustituirán por la categoría `Desconocido`. De esta forma se conserva la información de que esos registros no tenían un valor informado, sin asignarlos artificialmente a una categoría existente.

### 4.1 Imputación de la variable numérica 'horas_sueno'

In [7]:
# Calculamos la mediana de la variable horas_sueno
mediana_horas_sueno = df_clean["horas_sueno"].median()

# Sustituimos los valores nulos de horas_sueno por la mediana
df_clean["horas_sueno"] = df_clean["horas_sueno"].fillna(mediana_horas_sueno)

# Mostramos el valor utilizado para la imputación
print(f"Mediana utilizada para imputar horas_sueno: {mediana_horas_sueno:.2f}")

Mediana utilizada para imputar horas_sueno: 7.02


#### Imputación de horas_sueno

La variable `horas_sueno` es una variable numérica, por lo que se ha decidido imputar sus valores nulos utilizando la mediana.

Se utiliza la mediana en lugar de la media porque es menos sensible a posibles valores extremos. Esto permite reemplazar los datos ausentes con un valor representativo sin alterar demasiado la distribución original de la variable.

### 4.2 Imputación de variables categóricas

In [8]:
# Definimos las columnas categóricas que contienen valores nulos
columnas_categoricas_nulos = [
    "horario_estudio_preferido",
    "estilo_aprendizaje"
]

# Sustituimos los valores nulos por la categoría "Desconocido"
df_clean[columnas_categoricas_nulos] = df_clean[columnas_categoricas_nulos].fillna("Desconocido")

### Imputación de variables categóricas

Las variables `horario_estudio_preferido` y `estilo_aprendizaje` son categóricas y contenían valores ausentes.

En este caso, se ha optado por sustituir los nulos por la categoría `Desconocido`. Esta opción permite mantener todos los registros del dataset y, al mismo tiempo, diferenciar los casos en los que la información no estaba disponible.

También se podría haber sustituido por la moda, pero hemos preferido no aumentar la frecuencia de la categoría mayoritaria. 


### 4.3 Comprobación de valores nulos después de la imputación

In [9]:
# Comprobación de valores nulos después del tratamiento
valores_nulos_post = pd.DataFrame({
    "valores_nulos": df_clean.isnull().sum(),
    "porcentaje_nulos": (df_clean.isnull().mean() * 100).round(2)
})

# Mostramos únicamente las columnas que todavía tengan valores nulos
valores_nulos_post[valores_nulos_post["valores_nulos"] > 0]

,valores_nulos,porcentaje_nulos


### Comprobación tras el tratamiento de nulos

Después de aplicar la imputación, se vuelve a comprobar si quedan valores nulos en el dataset. Esta comprobación permite verificar que el tratamiento se ha aplicado correctamente antes de continuar con el resto del preprocesamiento.

Como se puede ver, tras el tratamiento aplicado, no quedan valores nulos en el dataset. Por tanto, el conjunto de datos ya está preparado para continuar con las siguientes comprobaciones del preprocesamiento.

### 4.4 Comprobación de categorías después de imputar

In [10]:
# Comprobamos las categorías de las variables categóricas imputadas
for columna in columnas_categoricas_nulos:
    print("=" * 50)
    print(f"Categorías de {columna}:")
    print(df_clean[columna].value_counts())
    print()

Categorías de horario_estudio_preferido:
horario_estudio_preferido
Noche          344
Tarde          337
Mañana         219
Desconocido    100
Name: count, dtype: int64

Categorías de estilo_aprendizaje:
estilo_aprendizaje
Visual               363
Auditivo             254
Kinestésico          178
Lectura/Escritura    155
Desconocido           50
Name: count, dtype: int64



### Revisión de categorías imputadas

Se revisan las categorías de las variables categóricas imputadas para comprobar que la categoría `Desconocido` se ha incorporado correctamente.

Esta revisión ayuda a confirmar que los valores ausentes han sido tratados de forma explícita y que no se han perdido registros durante el proceso.

## 5. Revisión de registros duplicados

Después de tratar los valores nulos, se revisa si existen registros duplicados en el dataset.

Los duplicados pueden afectar al entrenamiento de los modelos, ya que podrían hacer que determinados registros tengan más peso del que deberían.

En caso de detectar filas duplicadas, se valoraría su eliminación. Si no existen duplicados, el dataset puede continuar sin modificaciones en este punto.

### 5.1 Comprobación de duplicados

In [11]:
# Comprobación de registros duplicados en el dataset limpio
duplicados = df_clean.duplicated().sum()

print(f"Número de registros duplicados: {duplicados}")

Número de registros duplicados: 0


No se detectan registros duplicados en el dataset después del tratamiento de valores nulos.

Por tanto, no es necesario eliminar filas en este paso y se mantiene el mismo número de registros que en el dataset original.

### 5.2 Comprobación dimensiones del dataset tras la revisión

In [12]:
# Comprobación de dimensiones después de revisar duplicados
print(f"Dimensiones del dataset original: {df.shape}")
print(f"Dimensiones del dataset limpio: {df_clean.shape}")

Dimensiones del dataset original: (1000, 11)
Dimensiones del dataset limpio: (1000, 11)


### Comprobación de dimensiones

Se comparan las dimensiones del dataset original y del dataset limpio para verificar que no se han eliminado registros durante esta fase.

Esta comprobación permite confirmar que el tratamiento aplicado hasta este punto no ha alterado el número de filas ni columnas del dataset, salvo las modificaciones realizadas sobre los valores ausentes.

## 6. Revisión de tipos de datos

Después del tratamiento de valores nulos y la revisión de duplicados, se comprueban de nuevo los tipos de datos del dataset.

Este paso permite verificar que las variables numéricas y categóricas mantienen una estructura adecuada para las siguientes fases del proyecto.

Aunque en este notebook no se realizará todavía la codificación de variables categóricas ni el escalado, es importante confirmar que el dataset limpio conserva correctamente la información original.

### 6.1 Visualización de tipos de datos

In [13]:
# Revisión de los tipos de datos después del preprocesamiento inicial
df_clean.dtypes

horas_estudio_semanal        float64
nota_anterior                float64
tasa_asistencia              float64
horas_sueno                  float64
edad                           int64
nivel_dificultad              object
tiene_tutor                   object
horario_estudio_preferido     object
estilo_aprendizaje            object
nota_final                   float64
aprobado                       int64
dtype: object

### Tipos de datos después del preprocesamiento

Se revisan los tipos de datos de cada columna tras aplicar el tratamiento de valores nulos.

Las variables numéricas deben mantenerse como `int64` o `float64`, mientras que las variables categóricas deben conservarse como tipo `object`.

Esta comprobación ayuda a evitar problemas en fases posteriores, especialmente cuando se preparen los datos para los modelos de Machine Learning.

### 6.2 Separación de variables númericas y categóricas

In [14]:
# Identificación de variables numéricas y categóricas después de la limpieza
variables_numericas_clean = df_clean.select_dtypes(include=["int64", "float64"]).columns.tolist()
variables_categoricas_clean = df_clean.select_dtypes(include=["object"]).columns.tolist()

print("Variables numéricas:")
print(variables_numericas_clean)

print("\nVariables categóricas:")
print(variables_categoricas_clean)

Variables numéricas:
['horas_estudio_semanal', 'nota_anterior', 'tasa_asistencia', 'horas_sueno', 'edad', 'nota_final', 'aprobado']

Variables categóricas:
['nivel_dificultad', 'tiene_tutor', 'horario_estudio_preferido', 'estilo_aprendizaje']


### Identificación de variables numéricas y categóricas

Se separan las variables numéricas y categóricas del dataset limpio.

Esta separación será útil en el notebook de modelado, ya que cada tipo de variable requerirá un tratamiento diferente.

Las variables numéricas podrán utilizarse directamente en los modelos, aunque algunas podrían escalarse. Las variables categóricas deberán transformarse mediante técnicas de codificación antes de ser utilizadas por los algoritmos.


### 6.3 Revisión de valores únicos en variables categóricas

In [15]:
# Revisión de valores únicos en las variables categóricas
for columna in variables_categoricas_clean:
    print("=" * 60)
    print(f"Variable: {columna}")
    print(df_clean[columna].unique())
    print()

Variable: nivel_dificultad
['Fácil' 'Difícil' 'Medio']

Variable: tiene_tutor
['Sí' 'No']

Variable: horario_estudio_preferido
['Tarde' 'Mañana' 'Desconocido' 'Noche']

Variable: estilo_aprendizaje
['Lectura/Escritura' 'Desconocido' 'Visual' 'Auditivo' 'Kinestésico']



### Revisión de valores únicos en variables categóricas

Se revisan los valores únicos de cada variable categórica para comprobar que las categorías son coherentes.

Este paso permite detectar posibles errores de escritura, valores inesperados o categorías creadas durante el tratamiento de nulos.

En este caso, se espera encontrar la categoría `Desconocido` en las variables donde se habían imputado valores ausentes.

### 6.4 Revisión final de columnas del dataset limpio

In [16]:
# Revisión final de columnas del dataset limpio
df_clean.columns.tolist()


['horas_estudio_semanal',
 'nota_anterior',
 'tasa_asistencia',
 'horas_sueno',
 'edad',
 'nivel_dificultad',
 'tiene_tutor',
 'horario_estudio_preferido',
 'estilo_aprendizaje',
 'nota_final',
 'aprobado']

### Revisión final de columnas

Se revisa la lista final de columnas del dataset limpio.

En esta fase no se eliminan variables ni se crean nuevas columnas, ya que el objetivo principal del notebook es generar una versión limpia del dataset original.

Las transformaciones específicas para los modelos, como la codificación de variables categóricas o la separación entre variables predictoras y objetivo, se realizarán posteriormente en el notebook de modelado.

### 6.5 Comprobación final del dataset

In [17]:
# Comprobación final del dataset limpio antes de guardarlo
print(f"Filas del dataset limpio: {df_clean.shape[0]}")
print(f"Columnas del dataset limpio: {df_clean.shape[1]}")
print(f"Valores nulos totales: {df_clean.isnull().sum().sum()}")
print(f"Registros duplicados: {df_clean.duplicated().sum()}")

Filas del dataset limpio: 1000
Columnas del dataset limpio: 11
Valores nulos totales: 0
Registros duplicados: 0


### Comprobación final antes del guardado

Antes de guardar el dataset procesado, se realiza una comprobación final del estado de los datos.

Se revisa el número de filas, el número de columnas, la cantidad total de valores nulos y el número de registros duplicados.

Esta revisión permite confirmar que el dataset está preparado para guardarse como versión limpia y utilizarse en las siguientes fases del proyecto.

### 6.6 Conclusión de la revisión de los tipos de datos

### 6.6 Conclusión de la revisión de tipos de datos

Tras la revisión realizada, el dataset limpio mantiene una estructura adecuada.

Las variables numéricas y categóricas conservan tipos de datos coherentes con su contenido. Además, se confirma que las variables categóricas contienen categorías consistentes, incluyendo la categoría `Desconocido` incorporada durante el tratamiento de valores nulos.

Por tanto, el dataset está preparado para ser guardado como versión procesada y utilizado posteriormente en la fase de modelado.

## 7. Guardado del dataset procesado

Una vez finalizadas las comprobaciones principales del preprocesamiento, se guarda una versión limpia del dataset.

Este archivo procesado se almacenará en la carpeta `Data/processed` y será utilizado posteriormente en el notebook de modelado.

Guardar una versión procesada permite separar claramente los datos originales de los datos preparados, manteniendo una estructura de proyecto más ordenada y reproducible.

In [18]:
# Escribimos la ruta de salida para guardar el dataset limpio
ruta_processed = Path("../Data/processed/dataset_estudiantes_limpio.csv")

# Guardamos el dataset limpio en la carpeta Data/processed
df_clean.to_csv(ruta_processed, index=False)

print(f"Dataset procesado guardado correctamente en: {ruta_processed}")

Dataset procesado guardado correctamente en: ..\Data\processed\dataset_estudiantes_limpio.csv


### Guardado del archivo procesado

Se guarda el DataFrame limpio `df_clean` en formato CSV dentro de la carpeta `Data/processed`.

Se utiliza `index=False` para evitar que pandas añada una columna adicional con el índice del DataFrame.

Este archivo será la base para las siguientes fases del proyecto, especialmente para la preparación de variables, codificación, división train-test y entrenamiento de los modelos.

### 7.1 Comprobación que el archivo se ha guardado correctamente

In [19]:
# Comprobamos que el archivo procesado puede cargarse correctamente
df_processed_check = pd.read_csv(ruta_processed)

# Mostramos las primeras filas del archivo procesado
df_processed_check.head()

,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,nivel_dificultad,tiene_tutor,horario_estudio_preferido,estilo_aprendizaje,nota_final,aprobado
0,8.957476,48.830601,86.640182,6.675694,25,Fácil,Sí,Tarde,Lectura/Escritura,84.4,1
1,11.042524,80.825707,83.449655,4.616844,18,Difícil,No,Tarde,Desconocido,72.0,1
2,4.510776,90.383694,74.623607,7.755246,25,Fácil,No,Mañana,Lectura/Escritura,80.0,1
3,6.647213,81.878257,82.849841,8.592826,23,Fácil,No,Desconocido,Visual,78.2,1
4,1.000000,66.254179,54.539935,6.671840,21,Medio,No,Desconocido,Auditivo,66.0,1


### Comprobación de carga del archivo procesado

Después de guardar el dataset limpio, se vuelve a cargar el archivo desde la carpeta `Data/processed`.

Esta comprobación permite verificar que el archivo se ha generado correctamente y que puede utilizarse sin problemas en los siguientes notebooks del proyecto.


### 7.2 Comprobación dimensiones

In [20]:
# Comparación de dimensiones entre el dataset original, limpio y procesado guardado
print(f"Dataset original: {df.shape}")
print(f"Dataset limpio: {df_clean.shape}")
print(f"Dataset procesado cargado: {df_processed_check.shape}")

Dataset original: (1000, 11)
Dataset limpio: (1000, 11)
Dataset procesado cargado: (1000, 11)


### Comparación de dimensiones

Se comparan las dimensiones del dataset original, del dataset limpio y del archivo procesado cargado desde disco.

Esta comprobación permite confirmar que el archivo guardado conserva el mismo número de filas y columnas que el DataFrame procesado en memoria.

En este caso, el número de registros se mantiene porque no se han eliminado filas durante el preprocesamiento.

### 7.3 Comprobación final de valores nulos y duplicados

In [21]:
# Comprobación final del archivo procesado guardado
print(f"Valores nulos totales en el archivo procesado: {df_processed_check.isnull().sum().sum()}")
print(f"Registros duplicados en el archivo procesado: {df_processed_check.duplicated().sum()}")

Valores nulos totales en el archivo procesado: 0
Registros duplicados en el archivo procesado: 0


### Validación final del archivo procesado

Como última comprobación, se revisa que el archivo procesado guardado no contenga valores nulos ni registros duplicados.

Esta validación confirma que el dataset está preparado para ser utilizado en el siguiente notebook del proyecto.

### 7.4 Conclusión del guardado del dataset procesado

El dataset limpio se ha guardado correctamente en la carpeta `Data/processed`.

También se ha comprobado que el archivo puede cargarse de nuevo sin problemas y que conserva las dimensiones esperadas.

Además, el archivo procesado no contiene valores nulos ni registros duplicados, por lo que queda preparado para utilizarse en la siguiente fase del proyecto.

## 8. Conclusiones del preprocesamiento inicial

En este notebook se ha realizado una primera limpieza del dataset de estudiantes con el objetivo de dejar preparada una versión procesada para las siguientes fases del proyecto.

Se partió del archivo original almacenado en `Data/raw` y se creó una copia de trabajo llamada `df_clean` para no modificar directamente los datos originales.

El tratamiento principal aplicado ha sido la imputación de valores nulos. En el caso de `horas_sueno`, al tratarse de una variable numérica, se sustituyeron los valores ausentes por la mediana. Para las variables categóricas `horario_estudio_preferido` y `estilo_aprendizaje`, los valores nulos se reemplazaron por la categoría `Desconocido`.

Tras aplicar estos cambios, se comprobó que el dataset ya no contenía valores nulos. También se revisó la existencia de duplicados y no fue necesario eliminar registros, por lo que el dataset mantiene las mismas dimensiones que el archivo original.

Además, se revisaron los tipos de datos y las categorías de las variables categóricas para confirmar que la estructura del dataset continuaba siendo coherente después de la limpieza.

Finalmente, se guardó el archivo procesado en `Data/processed/dataset_estudiantes_limpio.csv`. Este archivo será la base para el siguiente notebook, donde se prepararán los datos para entrenar los modelos de regresión lineal y regresión logística.

En esta fase todavía no se han aplicado transformaciones como One-Hot Encoding, escalado o división train-test, ya que esas operaciones se realizarán de forma controlada durante el modelado.